# Semi-Supervised Multi-Task Learning on Multi-Digit MNIST
**Specialist (ResNet18) -> soft labels -> Generalist (ResNet50), with Pareto exploration/expansion and PCGrad / MGDA / FAMO**

Pipeline
1. **Data**: build a MultiDigitMNIST-style set (several small, overlapping 28x28 digits on a 64x64 canvas, like [shaohua0116/MultiDigitMNIST](https://github.com/shaohua0116/MultiDigitMNIST)). Each digit position = one task (3 tasks).
2. **Specialist (ResNet18)**: trained on plain MNIST digits *padded* to a 64x64 canvas so the digit size matches MultiMNIST. It has 2 objectives (clean digit / hard digit with overlapping neighbours) -> Pareto exploration + expansion.
3. **Soft labels**: the chosen specialist labels every digit slot of the (mostly unlabeled) MultiMNIST training images.
4. **Generalist (ResNet50)**: one shared backbone, 3 heads (multi-task), trained with knowledge distillation from the specialist's soft labels + a few hard labels (semi-supervised). Trained with LS / PCGrad / MGDA / FAMO under different preference vectors -> Pareto front, then expanded between neighbouring front points.

In [ ]:
# !pip install torch torchvision matplotlib numpy
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, matplotlib.pyplot as plt, random, time, copy
from torchvision import datasets, models

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', dev)

CFG = dict(
    N_TRAIN_MULTI = 20000,   # unlabeled-ish MultiMNIST train images
    N_TEST_MULTI  = 5000,
    N_SPEC_HARD   = 20000,   # specialist "hard" (overlapped crop) samples built from labeled MNIST
    LABELED_FRAC  = 0.05,    # fraction of MultiMNIST train that keeps hard labels (semi-supervised)
    BS = 128, LR = 1e-3,
    SPEC_EPOCHS = 3, GEN_EPOCHS = 4,
    T_KD = 2.0,              # distillation temperature
    LAM  = 0.7,              # weight KD vs CE on labeled subset
    METHODS = ['ls', 'pcgrad', 'mgda', 'famo'],
    N_PREF_EXTRA = 0,        # extra random preference vectors on top of uniform + vertex-biased ones
    EXPAND_ROUNDS = 1,
    SEED = 0,
)
# Tip: for a fast smoke test use N_TRAIN_MULTI=4000, *_EPOCHS=1, METHODS=['ls','pcgrad']

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG['SEED'])

## 1. Data: MNIST (specialist) and MultiDigitMNIST-style (generalist)

In [ ]:
CANVAS, DS, K = 64, 28, 3          # canvas size, digit size, digits per image
BASE_X = [0, 18, 36]               # left / middle / right slots (they overlap, like MultiMNIST)

tr = datasets.MNIST('./data', train=True,  download=True)
te = datasets.MNIST('./data', train=False, download=True)
mn_tr_x, mn_tr_y = tr.data.numpy(), tr.targets.numpy()
mn_te_x, mn_te_y = te.data.numpy(), te.targets.numpy()

def make_multi(imgs, labels, n, seed):
    # n images, each with K small digits; task k = class of digit in slot k
    rng = np.random.RandomState(seed)
    X = np.zeros((n, CANVAS, CANVAS), np.uint8); Y = np.zeros((n, K), np.int64); P = np.zeros((n, K, 2), np.int64)
    for i in range(n):
        idx = rng.randint(0, len(imgs), K)
        for k, j in enumerate(idx):
            x = int(np.clip(BASE_X[k] + rng.randint(-3, 4), 0, CANVAS - DS))
            y = rng.randint(4, CANVAS - DS - 4 + 1)
            X[i, y:y+DS, x:x+DS] = np.maximum(X[i, y:y+DS, x:x+DS], imgs[j])
            Y[i, k] = labels[j]; P[i, k] = (y, x)
    return X, Y, P

def crop_pad(X, P, k):
    # crop the k-th digit slot and paste it centered on a clean canvas (= specialist input format)
    out = np.zeros_like(X); off = (CANVAS - DS) // 2
    for i in range(len(X)):
        y, x = P[i, k]; out[i, off:off+DS, off:off+DS] = X[i, y:y+DS, x:x+DS]
    return out

def pad_mnist(imgs):
    out = np.zeros((len(imgs), CANVAS, CANVAS), np.uint8); off = (CANVAS - DS) // 2
    out[:, off:off+DS, off:off+DS] = imgs
    return out

# Generalist data
Xg_tr, Yg_tr, Pg_tr = make_multi(mn_tr_x, mn_tr_y, CFG['N_TRAIN_MULTI'], seed=1)
Xg_te, Yg_te, Pg_te = make_multi(mn_te_x, mn_te_y, CFG['N_TEST_MULTI'],  seed=2)
# Specialist data: clean padded MNIST  +  "hard" crops (digit with overlapping neighbours) from labeled MNIST train
Xs_c = pad_mnist(mn_tr_x); Ys_c = mn_tr_y
n_h = CFG['N_SPEC_HARD'] // K + 1
Xh_raw, Yh_raw, Ph_raw = make_multi(mn_tr_x, mn_tr_y, n_h, seed=3)
Xs_h = np.concatenate([crop_pad(Xh_raw, Ph_raw, k) for k in range(K)])[:CFG['N_SPEC_HARD']]
Ys_h = np.concatenate([Yh_raw[:, k] for k in range(K)])[:CFG['N_SPEC_HARD']]
nS = min(len(Xs_c), len(Xs_h)); Xs_c, Ys_c, Xs_h, Ys_h = Xs_c[:nS], Ys_c[:nS], Xs_h[:nS], Ys_h[:nS]
# Specialist test sets
Xs_c_te = pad_mnist(mn_te_x); Ys_c_te = mn_te_y
Xs_h_te = np.concatenate([crop_pad(Xg_te, Pg_te, k) for k in range(K)]); Ys_h_te = np.concatenate([Yg_te[:, k] for k in range(K)])

T_ = lambda a, dt=torch.long: torch.tensor(a, dtype=dt if dt == torch.long else dt, device=dev)
Xg_tr_t, Yg_tr_t, Xg_te_t, Yg_te_t = [torch.tensor(a).to(dev) for a in (Xg_tr, Yg_tr, Xg_te, Yg_te)]
Xs_c_t, Ys_c_t, Xs_h_t, Ys_h_t = [torch.tensor(a).to(dev) for a in (Xs_c, Ys_c, Xs_h, Ys_h)]
Xs_c_te_t, Ys_c_te_t, Xs_h_te_t, Ys_h_te_t = [torch.tensor(a).to(dev) for a in (Xs_c_te, Ys_c_te, Xs_h_te, Ys_h_te)]

fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    ax[0, i].imshow(Xg_tr[i], cmap='gray'); ax[0, i].set_title('multi: ' + ''.join(map(str, Yg_tr[i]))); ax[0, i].axis('off')
    ax[1, i].imshow(Xs_c[i], cmap='gray'); ax[1, i].set_title(f'specialist: {Ys_c[i]}'); ax[1, i].axis('off')
plt.show()

## 2. Models (ResNet18 specialist / ResNet50 generalist, shared backbone + task heads)

In [ ]:
class MTLNet(nn.Module):
    def __init__(self, arch, n_heads, n_cls=10):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)   # grayscale input
        feat = bb.fc.in_features; bb.fc = nn.Identity()
        self.backbone = bb
        self.heads = nn.ModuleList([nn.Linear(feat, n_cls) for _ in range(n_heads)])
    def forward(self, x):
        f = self.backbone(x); return [h(f) for h in self.heads]

def prep(xb, shift=0):
    x = xb.float().div(255.).sub(0.13).div(0.31).unsqueeze(1)
    if shift:
        x = torch.roll(x, (random.randint(-shift, shift), random.randint(-shift, shift)), (2, 3))
    return x

@torch.no_grad()
def predict(model, X, bs=512, temp=1.0):
    model.eval(); outs = None
    for s in range(0, len(X), bs):
        o = model(prep(X[s:s+bs]))
        o = [F.softmax(z / temp, -1).cpu() for z in o]
        outs = [[a] for a in o] if outs is None else [p + [a] for p, a in zip(outs, o)]
    return [torch.cat(p) for p in outs]   # list over heads of (n, 10)

## 3. Multi-objective gradient methods: PCGrad, MGDA, FAMO
All operate on the **shared backbone** gradients; task heads always get their own gradient. `pref` is a preference vector that re-weights task losses, which is what moves the solution along the Pareto front.

In [ ]:
def flat_grads(losses, params):
    gs = []
    for l in losses:
        g = torch.autograd.grad(l, params, retain_graph=True, allow_unused=True)
        gs.append(torch.cat([(gi if gi is not None else torch.zeros_like(p)).flatten() for gi, p in zip(g, params)]))
    return torch.stack(gs)                                   # (T, P)

def pcgrad(G):
    # project each task gradient away from conflicting ones (Yu et al. 2020)
    T = len(G); out = G.clone()
    for i in range(T):
        for j in random.sample(range(T), T):
            if j == i: continue
            dot = (out[i] * G[j]).sum()
            if dot < 0: out[i] = out[i] - dot / (G[j].norm() ** 2 + 1e-12) * G[j]
    return out.sum(0)

def mgda_weights(G, iters=50):
    # min-norm point in the convex hull of task gradients (Desideri 2012; Sener & Koltun 2018), Frank-Wolfe solver
    M = G @ G.T; T = len(G); a = torch.ones(T, device=G.device) / T
    for _ in range(iters):
        i = torch.argmin(M @ a); e = torch.zeros_like(a); e[i] = 1; d = e - a
        gamma = torch.clamp(-(d @ M @ a) / (d @ M @ d + 1e-12), 0, 1)
        a = a + gamma * d
    return a

class FAMO:
    # Fast Adaptive Multitask Optimization (Liu et al. 2023): O(1) space/time, balances rate of loss decrease
    def __init__(self, T, pref=None, lr=0.025, gamma=1e-3):
        init = torch.log(torch.tensor(pref if pref is not None else [1.0 / T] * T, dtype=torch.float32))
        self.w = init.clone().to(dev).requires_grad_(True)
        self.opt = torch.optim.Adam([self.w], lr=lr, weight_decay=gamma); self.eps = 1e-8
        self.min_loss = torch.zeros(T, device=dev)
    def weighted(self, losses):
        D = losses - self.min_loss + self.eps; z = F.softmax(self.w, -1); c = (z / D).sum().detach()
        return (D.log() * z / c).sum()
    def update(self, prev, curr):
        delta = (prev - self.min_loss + self.eps).log() - (curr - self.min_loss + self.eps).log()
        with torch.enable_grad():
            d = torch.autograd.grad(F.softmax(self.w, -1), self.w, grad_outputs=delta.detach())[0]
        self.opt.zero_grad(); self.w.grad = d; self.opt.step()

def train_mtl(model, loss_fn, n_train, T, method, pref, epochs, lr=None, bs=None):
    lr = lr or CFG['LR']; bs = bs or CFG['BS']
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * (n_train // bs); sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    shared = list(model.backbone.parameters()); pref_t = torch.tensor(pref, device=dev, dtype=torch.float32) * T
    famo = FAMO(T, pref) if method == 'famo' else None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n_train, device=dev)
        for s in range(0, n_train - bs + 1, bs):
            idx = perm[s:s+bs]; losses = loss_fn(model, idx); opt.zero_grad()
            if method == 'ls':
                sum(pref_t[i] * losses[i] for i in range(T)).backward()
            elif method == 'famo':
                famo.weighted(torch.stack(losses)).backward()
            else:   # pcgrad / mgda
                wl = [pref_t[i] * losses[i] for i in range(T)]
                G = flat_grads(wl, shared)
                sum(wl).backward()                                  # gives head grads (shared grads overwritten next)
                g = pcgrad(G) if method == 'pcgrad' else (mgda_weights(G) @ G) * T
                o = 0
                for p in shared:
                    n = p.numel(); p.grad = g[o:o+n].view_as(p).clone(); o += n
            opt.step(); sched.step()
            if famo is not None:
                with torch.no_grad(): new = torch.stack(loss_fn(model, idx))
                famo.update(torch.stack(losses).detach(), new)
    return model

## 4. Pareto exploration + expansion (generic, used for both specialist and generalist)
* **Exploration**: for each method, train with several preference vectors (uniform + vertex-biased + optional random). Each run gives a vector of per-task test errors.
* **Expansion**: take the non-dominated points, and for neighbouring front points of the same method train with the *midpoint preference* to fill gaps in the front.

In [ ]:
def make_prefs(T, n_extra=0, seed=0):
    prefs = [[1.0 / T] * T]
    for k in range(T):
        prefs.append([0.6 if i == k else 0.4 / (T - 1) for i in range(T)])
    rng = np.random.RandomState(seed)
    for _ in range(n_extra): prefs.append(list(rng.dirichlet(np.ones(T))))
    return prefs

def nondominated(E):
    E = np.array(E); mask = np.ones(len(E), bool)
    for i in range(len(E)):
        for j in range(len(E)):
            if i != j and np.all(E[j] <= E[i]) and np.any(E[j] < E[i]): mask[i] = False; break
    return mask

def pareto_explore(run_fn, methods, prefs, tag=''):
    res = []
    for m in methods:
        for p in prefs:
            t0 = time.time(); r = run_fn(m, p); r.update(method=m, pref=p); res.append(r)
            print(f'[{tag}] {m:7s} pref={np.round(p,2)} err={np.round(r["err"],4)}  ({time.time()-t0:.0f}s)')
    return res

def pareto_expand(res, run_fn, rounds=1, tag=''):
    for rd in range(rounds):
        mask = nondominated([r['err'] for r in res]); front = [r for r, k in zip(res, mask) if k]
        new = []
        for m in set(r['method'] for r in front):
            fm = sorted([r for r in front if r['method'] == m], key=lambda r: r['err'][0])
            for a, b in zip(fm[:-1], fm[1:]):
                p = ((np.array(a['pref']) + np.array(b['pref'])) / 2).tolist()
                if not any(np.allclose(p, r['pref']) and r['method'] == m for r in res):
                    new.append((m, p))
        print(f'[{tag}] expansion round {rd+1}: {len(new)} new preference points')
        for m, p in new:
            r = run_fn(m, p); r.update(method=m, pref=p, expanded=True); res.append(r)
            print(f'[{tag}] {m:7s} pref={np.round(p,2)} err={np.round(r["err"],4)} (expanded)')
    return res

def plot_pareto(res, names, title):
    E = np.array([r['err'] for r in res]); mask = nondominated(E); T = E.shape[1]
    pairs = [(i, j) for i in range(T) for j in range(i + 1, T)]
    fig, axs = plt.subplots(1, len(pairs), figsize=(5.2 * len(pairs), 4.4)); axs = np.atleast_1d(axs)
    cols = dict(ls='tab:gray', pcgrad='tab:blue', mgda='tab:green', famo='tab:red')
    for ax, (i, j) in zip(axs, pairs):
        for m in cols:
            sel = [k for k, r in enumerate(res) if r['method'] == m]
            if sel: ax.scatter(E[sel, i], E[sel, j], c=cols[m], label=m, s=[60 if res[k].get('expanded') else 35 for k in sel],
                               marker='o', alpha=.8)
        F_ = E[mask][:, [i, j]]; F_ = F_[np.argsort(F_[:, 0])]
        ax.plot(F_[:, 0], F_[:, 1], 'k--', lw=1, label='Pareto front'); ax.set_xlabel(f'error {names[i]}'); ax.set_ylabel(f'error {names[j]}')
        ax.grid(alpha=.3)
    axs[0].legend(); fig.suptitle(title); plt.tight_layout(); plt.show()
    return mask

## 5. Stage 1: Specialist ResNet18 (MNIST padded) with Pareto exploration
Two objectives on a shared ResNet18: **head 0** = clean padded MNIST digit; **head 1** = hard digit crop (neighbouring digits overlapping, like inside MultiMNIST). The specialist should be good at both, and the Pareto front shows the trade-off.

In [ ]:
def spec_loss_fn(model, idx):
    oc = model(prep(Xs_c_t[idx], shift=2))[0]; oh = model(prep(Xs_h_t[idx], shift=2))[1]
    return [F.cross_entropy(oc, Ys_c_t[idx]), F.cross_entropy(oh, Ys_h_t[idx])]

def spec_eval(model):
    pc = predict(model, Xs_c_te_t)[0].argmax(1).to(dev); ph = predict(model, Xs_h_te_t)[1].argmax(1).to(dev)
    return [1 - (pc == Ys_c_te_t).float().mean().item(), 1 - (ph == Ys_h_te_t).float().mean().item()]

def spec_run(method, pref):
    set_seed(CFG['SEED']); m = MTLNet('resnet18', 2).to(dev)
    train_mtl(m, spec_loss_fn, len(Xs_c_t), 2, method, pref, CFG['SPEC_EPOCHS'])
    return dict(err=spec_eval(m), state={k: v.cpu() for k, v in m.state_dict().items()})

spec_prefs = [[0.5, 0.5], [0.8, 0.2], [0.2, 0.8]]
spec_res = pareto_explore(spec_run, CFG['METHODS'], spec_prefs, 'specialist')
spec_res = pareto_expand(spec_res, spec_run, CFG['EXPAND_ROUNDS'], 'specialist')
mask = plot_pareto(spec_res, ['clean MNIST', 'hard crop'], 'Specialist (ResNet18) Pareto front')

In [ ]:
# choose the knee point of the specialist front = lowest mean error among non-dominated solutions
front = [r for r, k in zip(spec_res, mask) if k]
best_spec = min(front, key=lambda r: np.mean(r['err']))
print('chosen specialist:', best_spec['method'], np.round(best_spec['pref'], 2), 'errors (clean, hard):', np.round(best_spec['err'], 4))
specialist = MTLNet('resnet18', 2).to(dev); specialist.load_state_dict(best_spec['state']); specialist.eval()

## 6. Stage 2: Soft labels from the specialist for every digit slot
Each slot of each MultiMNIST training image is cropped, padded to the specialist's input format, and run through the specialist. Soft labels = mean of both heads' temperature-softmax. The hard labels are only kept for a small labeled subset (semi-supervised).

In [ ]:
Tkd = CFG['T_KD']
soft = []
for k in range(K):
    crops = torch.tensor(crop_pad(Xg_tr, Pg_tr, k)).to(dev)
    ph = predict(specialist, crops, temp=Tkd)               # list over heads (n,10)
    soft.append(torch.stack(ph).mean(0))                    # (n,10)
soft_t = torch.stack(soft, 1).to(dev)                       # (n, K, 10)

# diagnostics only (uses ground truth, never used for training)
teach_acc_tr = (soft_t.argmax(-1) == Yg_tr_t).float().mean(0)
print('teacher pseudo-label accuracy per slot on train:', teach_acc_tr.cpu().numpy().round(4))
te_soft = torch.stack([torch.stack(predict(specialist, torch.tensor(crop_pad(Xg_te, Pg_te, k)).to(dev))).mean(0) for k in range(K)], 1)
teacher_err = (1 - (te_soft.argmax(-1).to(dev) == Yg_te_t).float().mean(0)).cpu().numpy()
print('teacher test error (oracle crops) per slot:', teacher_err.round(4))

g = torch.Generator().manual_seed(123)
lab_mask = (torch.rand(len(Xg_tr), generator=g) < CFG['LABELED_FRAC']).to(dev)
print('labeled MultiMNIST train images:', int(lab_mask.sum()), '/', len(lab_mask))

## 7. Stage 3: Generalist ResNet50 (multi-task, semi-supervised distillation)
Loss for task k: `LAM * T^2 * KL(student_T || specialist_soft) + (1-LAM) * CE(labeled subset only)`.
Only the multi-digit image is the generalist input (no crops, no oracle positions).

In [ ]:
def make_gen_loss(lam):
    def gen_loss_fn(model, idx):
        out = model(prep(Xg_tr_t[idx])); m = lab_mask[idx]; losses = []
        for k in range(K):
            kd = F.kl_div(F.log_softmax(out[k] / Tkd, -1), soft_t[idx, k], reduction='batchmean') * Tkd ** 2
            ce = F.cross_entropy(out[k][m], Yg_tr_t[idx, k][m]) if m.any() else out[k].sum() * 0
            losses.append(lam * kd + (1 - lam) * ce)
        return losses
    return gen_loss_fn

def gen_eval(model):
    p = predict(model, Xg_te_t); return [1 - (p[k].argmax(1).to(dev) == Yg_te_t[:, k]).float().mean().item() for k in range(K)]

def gen_run(method, pref, lam=None):
    set_seed(CFG['SEED']); m = MTLNet('resnet50', K).to(dev)
    train_mtl(m, make_gen_loss(CFG['LAM'] if lam is None else lam), len(Xg_tr_t), K, method, pref, CFG['GEN_EPOCHS'])
    return dict(err=gen_eval(m))

gen_prefs = make_prefs(K, CFG['N_PREF_EXTRA'])
gen_res = pareto_explore(gen_run, CFG['METHODS'], gen_prefs, 'generalist')
gen_res = pareto_expand(gen_res, gen_run, CFG['EXPAND_ROUNDS'], 'generalist')
gmask = plot_pareto(gen_res, [f'slot {k}' for k in range(K)], 'Generalist (ResNet50) Pareto front, semi-supervised KD')

## 8. Baselines and summary

In [ ]:
# Baseline: same ResNet50, labeled subset only (no soft labels, lam=0), uniform LS
base = gen_run('ls', [1.0 / K] * K, lam=0.0)
print('baseline (labeled-only) test error per slot:', np.round(base['err'], 4), 'mean', np.mean(base['err']).round(4))

import pandas as pd
rows = []
for m in CFG['METHODS']:
    sel = [r for r in gen_res if r['method'] == m]
    b = min(sel, key=lambda r: np.mean(r['err']))
    rows.append(dict(method=m, mean_err=np.mean(b['err']), **{f'slot{k}': b['err'][k] for k in range(K)}, pref=np.round(b['pref'], 2).tolist()))
rows.append(dict(method='baseline labeled-only', mean_err=np.mean(base['err']), **{f'slot{k}': base['err'][k] for k in range(K)}))
rows.append(dict(method='specialist teacher (oracle crops)', mean_err=teacher_err.mean(), **{f'slot{k}': teacher_err[k] for k in range(K)}))
print(pd.DataFrame(rows).round(4).to_string(index=False))

# Pareto quality: hypervolume-style proxy = number of non-dominated points per method
E = np.array([r['err'] for r in gen_res]); gm = nondominated(E)
print({m: int(sum(1 for r, k in zip(gen_res, gm) if k and r['method'] == m)) for m in CFG['METHODS']}, '<- non-dominated points per method')

### Notes
* **Soft-label flow**: teacher (ResNet18 on padded/cropped single digits) -> temperature soft labels -> student (ResNet50 on the full multi-digit image). Only `LABELED_FRAC` of images contribute the hard-label CE term.
* **Pareto exploration** = sweep of preference vectors per method; **expansion** = extra runs at midpoint preferences between neighbouring non-dominated solutions. The same code is applied to the specialist (2 objectives) and the generalist (3 objectives).
* **Methods**: `ls` linear scalarization baseline, `pcgrad` gradient surgery, `mgda` min-norm Pareto-stationary direction, `famo` adaptive loss weights (preference used as initial weights).
* Runtime: ResNet50 + PCGrad/MGDA needs T backward passes per step. Reduce `N_TRAIN_MULTI`, epochs, or `METHODS` in `CFG` for a quick run; a GPU is strongly recommended.
* To use the original MultiDigitMNIST files instead, replace `make_multi` with the repo's generated arrays (keep the slot positions `P` for the specialist crops).